In [1]:
# === Held-out evaluation for latest v2 run (macro/micro Dice, CSV+JSON) ===
from pathlib import Path
import importlib.util, json, time
import numpy as np

# --------- Paths you can tweak ----------
RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356")
TEST_DIR  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
T1_DIR    = TEST_DIR / "t1"
MSK_DIR   = TEST_DIR / "masks"
TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
# ---------------------------------------

# ---- Import training module (gives us loader/preproc utils + custom layers) ----
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# ---- Find model to load (prefer full .keras; else build and load best weights) ----
models_dir     = RUN_DIR / "models"
callbacks_dir  = RUN_DIR / "callbacks"
full_models    = sorted(models_dir.glob("*.keras"))
best_weights   = callbacks_dir / "best_model_dynamic.weights.h5"
cfg_json_path  = models_dir / "config.json"  # written by the training code

model = None
INPUT_SHAPE = None

# custom_objects for loading .keras
custom_objects = {
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
}

try:
    from keras.saving import load_model as keras_load_model
except Exception:
    from tensorflow.keras.models import load_model as keras_load_model

if full_models:
    model_path = full_models[-1]
    print(f"Loading FULL model: {model_path}")
    model = keras_load_model(model_path, compile=False, custom_objects=custom_objects)
    INPUT_SHAPE = tuple(model.input_shape[1:])
else:
    # Build a matching model from saved config.json, then load weights
    assert cfg_json_path.exists(), f"Missing {cfg_json_path}"
    with open(cfg_json_path) as f:
        saved_cfg = json.load(f)
    # Minimal fields needed to rebuild the same shapes
    cfg = seg.DynamicTrainingConfig(
        DATA_DIR=TEST_DIR,  # dummy root; we won't train
        MODEL_DIR=models_dir,
        CALLBACKS_DIR=callbacks_dir,
        INPUT_SHAPE=tuple(saved_cfg["INPUT_SHAPE"]) if saved_cfg.get("INPUT_SHAPE") else None,
        BASE_FILTERS=int(saved_cfg.get("BASE_FILTERS", 8)),
        SAM_HEADS=int(saved_cfg.get("SAM_HEADS", 2)),
    )
    if cfg.INPUT_SHAPE in (None, (), []):
        raise RuntimeError("INPUT_SHAPE missing in saved config; cannot rebuild model.")
    INPUT_SHAPE = cfg.INPUT_SHAPE
    print("Rebuilding model from config.json and loading best weights…")
    model = seg.build_dynamic_model(cfg)
    model.load_weights(str(best_weights))

print("INPUT_SHAPE:", INPUT_SHAPE)

# ---- Build the held-out list (use separated subfolders to avoid duplicates) ----
if T1_DIR.exists() and MSK_DIR.exists():
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR, IMAGES_DIR=T1_DIR, MASKS_DIR=MSK_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")
else:
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")

cfg_eval.INPUT_SHAPE = INPUT_SHAPE
pairs, lesion_presence = seg.load_generic_dataset(cfg_eval)
print(f"Pairs: {len(pairs)} | % non-empty masks: {lesion_presence.mean()*100:.1f}%")

# ---- Dice helpers ----
def dice_soft(y, p):
    y = y.astype(np.float64); p = p.astype(np.float64)
    inter = (y * p).sum()
    return (2.0*inter) / (y.sum() + p.sum() + 1e-12)

def dice_hard(y, p, th=0.5):
    pb = (p >= th).astype(np.float64)
    inter = (y*pb).sum()
    return (2.0*inter) / (y.sum() + pb.sum() + 1e-12)

# ---- Evaluate (macro: per-case mean; micro: global) ----
macro_softs, macro_hards = [], []
s_inter_soft = np.float64(0.0)
s_sumy_soft  = np.float64(0.0)
s_sump_soft  = np.float64(0.0)

TH = 0.50  # you can sweep later

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)

    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = img
    p = model.predict(x, verbose=0)[0,...,0].astype(np.float32)

    # per-case dice
    ds = dice_soft(y, p)
    dh = dice_hard(y, p, th=TH)
    macro_softs.append(ds); macro_hards.append(dh)

    # micro-soft accumulators
    s_inter_soft += (y.astype(np.float64) * p.astype(np.float64)).sum()
    s_sumy_soft  += y.sum(dtype=np.float64)
    s_sump_soft  += p.sum(dtype=np.float64)

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{TH:.2f}={dh:.4f}")

macro_soft = float(np.mean(macro_softs))
macro_hard = float(np.mean(macro_hards))
micro_soft = float((2.0*s_inter_soft) / (s_sumy_soft + s_sump_soft + 1e-12))

# compute micro-hard at the same threshold
# we need a second pass for exact global hard (memory-safe). Do a quick second pass:
s_inter_hard = np.float64(0.0)
s_sumy_hard  = np.float64(0.0)
s_sump_hard  = np.float64(0.0)
for img_p, msk_p in pairs:
    y = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    p = model.predict(x, verbose=0)[0,...,0]
    pb = (p >= TH).astype(np.float64)
    s_inter_hard += (y.astype(np.float64) * pb).sum()
    s_sumy_hard  += y.sum(dtype=np.float64)
    s_sump_hard  += pb.sum(dtype=np.float64)

micro_hard = float((2.0*s_inter_hard) / (s_sumy_hard + s_sump_hard + 1e-12))

# ---- Report + save ----
print("\n=== HELD-OUT RESULTS ===")
print(f"Per-case (macro) soft Dice      : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @ {TH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice        : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @ {TH:.2f}  : {micro_hard:.4f}")
print(f"Val set size: {len(pairs)} cases")

# Save into this run folder (so results travel with the model)
out_dir = RUN_DIR / "test_eval"
out_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")

# per-case CSV
csv_path = out_dir / f"test_metrics_{ts}.csv"
with open(csv_path, "w") as f:
    f.write("case,soft_dice,hard_dice_at_{:.2f}\n".format(TH))
    for (img_p, _), ds, dh in zip(pairs, macro_softs, macro_hards):
        f.write(f"{img_p.stem},{ds:.6f},{dh:.6f}\n")

# summary JSON
summary = {
    "threshold": TH,
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "n_cases": len(pairs),
    "run_dir": str(RUN_DIR),
}
json_path = out_dir / f"test_metrics_summary_{ts}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV ->", csv_path)
print("Wrote summary JSON ->", json_path)


2025-11-07 15:07:44.633454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1762553266.544921 1325164 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762553266.545935 1325164 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1762553266.546209 1325164 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1762553266.547177 1325164 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2025-11-07 15:07:46,608 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-07 15:07:46,609 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-07 15:07:46,609 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Loading FULL model: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356/models/smart_sota_dynamic_20251107_103356.keras


2025-11-07 15:07:48,058 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-07 15:07:48,059 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.99GB | GPU mem tracking failed | Disk: 1231.0GB free
2025-11-07 15:07:48,065 - SmartSOTA_Dynamic - INFO - 📂 Two-folder mode: images=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/t1), masks=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/masks)
2025-11-07 15:07:48,066 - SmartSOTA_Dynamic - INFO - Found 138 image files and 138 mask files


INPUT_SHAPE: (192, 224, 192, 1)


2025-11-07 15:08:06,687 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-07 15:08:06,688 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-07 15:08:06,689 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.00GB | GPU mem tracking failed | Disk: 1231.0GB free


Pairs: 138 | % non-empty masks: 100.0%


2025-11-07 15:08:07.729390: I external/local_xla/xla/service/service.cc:163] XLA service 0x70f828005410 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 15:08:07.729418: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-07 15:08:07.729424: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-07 15:08:07.828043: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-07 15:08:08.103343: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
I0000 00:00:1762553294.364745 1325315 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[10/138] last case soft=0.8195 hard@0.50=0.8237
[20/138] last case soft=0.8492 hard@0.50=0.8516
[30/138] last case soft=0.7897 hard@0.50=0.7932
[40/138] last case soft=0.5236 hard@0.50=0.5261
[50/138] last case soft=0.6927 hard@0.50=0.6957
[60/138] last case soft=0.6711 hard@0.50=0.6754
[70/138] last case soft=0.7172 hard@0.50=0.7217
[80/138] last case soft=0.4940 hard@0.50=0.5029
[90/138] last case soft=0.6561 hard@0.50=0.6604
[100/138] last case soft=0.0086 hard@0.50=0.0089
[110/138] last case soft=0.1217 hard@0.50=0.1236
[120/138] last case soft=0.2790 hard@0.50=0.2840
[130/138] last case soft=0.1385 hard@0.50=0.1425
[138/138] last case soft=0.0001 hard@0.50=0.0000

=== HELD-OUT RESULTS ===
Per-case (macro) soft Dice      : 0.4316
Per-case (macro) hard Dice @ 0.50: 0.4362
Global (micro) soft Dice        : 0.6286
Global (micro) hard Dice @ 0.50  : 0.6328
Val set size: 138 cases

Wrote per-case CSV -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_10

In [3]:
# --- Get INPUT_SHAPE from saved full model (or fall back) ---
from pathlib import Path
import json

RUN_DIR  = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356")
models_dir = RUN_DIR / "models"

# Try Keras loader first
try:
    from keras.saving import load_model as keras_load_model
except Exception:
    from tensorflow.keras.models import load_model as keras_load_model

# Load your training module (for custom_objects)
import importlib.util
TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# 1) Prefer a saved full model (*.keras)
model_files = sorted(models_dir.glob("*.keras"))
INPUT_SHAPE = None
if model_files:
    fm = model_files[-1]
    m = keras_load_model(fm, compile=False, custom_objects={
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention": seg.SAM2Attention,
        "CombinedLoss": seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss": seg.dice_loss,
        "boundary_loss": seg.boundary_loss,
    })
    INPUT_SHAPE = tuple(m.input_shape[1:])
    print("Loaded full model:", fm.name, "| INPUT_SHAPE:", INPUT_SHAPE)
else:
    # 2) Try config.json written during training
    cfg_json = models_dir / "config.json"
    if cfg_json.exists():
        with open(cfg_json) as f:
            cfg_payload = json.load(f)
        INPUT_SHAPE = tuple(cfg_payload.get("INPUT_SHAPE") or (192,224,192,1))
        print("No .keras found. Using INPUT_SHAPE from config.json:", INPUT_SHAPE)
    else:
        # 3) Final fallback: known shape
        INPUT_SHAPE = (192, 224, 192, 1)
        print("No model/config found. Falling back to:", INPUT_SHAPE)

# --- Build eval config & dataset (ensure consistent resampling) ---
from pathlib import Path
TEST_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
T1_DIR, MSK_DIR = TEST_DIR / "t1", TEST_DIR / "masks"

cfg_eval = seg.DynamicTrainingConfig(
    DATA_DIR=TEST_DIR,
    IMAGES_DIR=T1_DIR,
    MASKS_DIR=MSK_DIR,
    MODEL_DIR=RUN_DIR / "_tmp_models",
    CALLBACKS_DIR=RUN_DIR / "_tmp_callbacks",
)
cfg_eval.INPUT_SHAPE = INPUT_SHAPE
cfg_eval.RESAMPLE_TO_TARGET = True  # keep eval preprocessing identical to training

pairs, _ = seg.load_generic_dataset(cfg_eval)
print("Pairs loaded for eval:", len(pairs))


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-07 15:13:33,874 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-07 15:13:33,877 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-07 15:13:33,878 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-07 15:13:33,878 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2


Strategy: MirroredStrategy


2025-11-07 15:13:34,791 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-07 15:13:34,792 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=6.87GB | GPU mem tracking failed | Disk: 1231.0GB free
2025-11-07 15:13:34,797 - SmartSOTA_Dynamic - INFO - 📂 Two-folder mode: images=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/t1), masks=138 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires/masks)
2025-11-07 15:13:34,798 - SmartSOTA_Dynamic - INFO - Found 138 image files and 138 mask files


Loaded full model: smart_sota_dynamic_20251107_103356.keras | INPUT_SHAPE: (192, 224, 192, 1)


2025-11-07 15:13:53,954 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-07 15:13:53,955 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-07 15:13:53,956 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=6.90GB | GPU mem tracking failed | Disk: 1231.0GB free


Pairs loaded for eval: 138


In [4]:
# --- Per-case eval: soft/hard Dice, lesion voxels, mean pred; save CSV/JSON and show top/bottom cases ---

import numpy as np, json, time, csv
from pathlib import Path
import nibabel as nib

# Reuse objects if they exist; otherwise reload model from your run folder
try:
    m  # noqa: F821
except NameError:
    RUN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356")
    models_dir = RUN_DIR / "models"
    try:
        from keras.saving import load_model as keras_load_model
    except Exception:
        from tensorflow.keras.models import load_model as keras_load_model

    import importlib.util
    TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
    spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
    seg = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(seg)

    fm = sorted(models_dir.glob("*.keras"))[-1]
    m = keras_load_model(fm, compile=False, custom_objects={
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention": seg.SAM2Attention,
        "CombinedLoss": seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss": seg.dice_loss,
        "boundary_loss": seg.boundary_loss,
    })
    INPUT_SHAPE = tuple(m.input_shape[1:])
    print("Loaded model:", fm.name, "| INPUT_SHAPE:", INPUT_SHAPE)

# Ensure we have cfg_eval, pairs, INPUT_SHAPE
try:
    cfg_eval, pairs, INPUT_SHAPE  # noqa: F821
except NameError:
    TEST_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
    T1_DIR, MSK_DIR = TEST_DIR / "t1", TEST_DIR / "masks"
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                         IMAGES_DIR=T1_DIR,
                                         MASKS_DIR=MSK_DIR,
                                         MODEL_DIR=TEST_DIR/"_tmp_models",
                                         CALLBACKS_DIR=TEST_DIR/"_tmp_callbacks")
    cfg_eval.INPUT_SHAPE = tuple(m.input_shape[1:])
    cfg_eval.RESAMPLE_TO_TARGET = True
    pairs, _ = seg.load_generic_dataset(cfg_eval)
    print("Re-loaded pairs:", len(pairs))

def dice_soft(y, p):
    y = y.astype(np.float64); p = p.astype(np.float64)
    inter = (y * p).sum()
    return (2.0 * inter) / (y.sum() + p.sum() + 1e-12)

def dice_hard(y, p, th=0.5):
    pb = (p >= th).astype(np.float32)
    inter = (y * pb).sum(dtype=np.float64)
    return (2.0 * inter) / (y.sum(dtype=np.float64) + pb.sum(dtype=np.float64) + 1e-12)

# iterate and collect metrics
rows = []
s_inter = np.float64(0.0); s_sumy = np.float64(0.0); s_sump = np.float64(0.0)
for idx, (img_p, msk_p) in enumerate(pairs, 1):
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask (str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img
    p = m.predict(x, verbose=0)[0,...,0].astype(np.float32)

    ds = dice_soft(y, p)
    dh = dice_hard(y, p, th=0.50)
    vox = int((y > 0).sum())
    mp  = float(p.mean())

    # accumulate for micro soft
    s_inter += (y.astype(np.float64) * p.astype(np.float64)).sum()
    s_sumy  += y.sum(dtype=np.float64)
    s_sump  += p.sum(dtype=np.float64)

    rows.append({
        "idx": idx,
        "image": img_p.name,
        "mask": msk_p.name,
        "lesion_voxels": vox,
        "dice_soft": float(ds),
        "dice_hard@0.50": float(dh),
        "mean_pred": mp,
    })
    if idx % 10 == 0 or idx == len(pairs):
        print(f"[{idx}/{len(pairs)}] last case soft={ds:.4f} hard@0.50={dh:.4f}")

# summary
macro_soft = float(np.mean([r["dice_soft"] for r in rows]))
macro_hard = float(np.mean([r["dice_hard@0.50"] for r in rows]))
micro_soft = float((2.0 * s_inter) / (s_sumy + s_sump + 1e-12))

print("\n=== HELD-OUT SUMMARY ===")
print(f"Macro soft Dice        : {macro_soft:.4f}")
print(f"Macro hard Dice @0.50  : {macro_hard:.4f}")
print(f"Micro soft Dice        : {micro_soft:.4f}")
print(f"N cases: {len(rows)}")

# top/bottom by soft Dice
rows_sorted = sorted(rows, key=lambda r: r["dice_soft"])
print("\nBottom 10 (soft Dice):")
for r in rows_sorted[:10]:
    print(f"  {r['idx']:>3d} | {r['dice_soft']:.4f} | vox={r['lesion_voxels']:>7d} | {r['image']}")

print("\nTop 10 (soft Dice):")
for r in rows_sorted[-10:][::-1]:
    print(f"  {r['idx']:>3d} | {r['dice_soft']:.4f} | vox={r['lesion_voxels']:>7d} | {r['image']}")

# save CSV + JSON
RUN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356")
out_dir = RUN_DIR / "test_eval"; out_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M%S")
csv_path  = out_dir / f"test_metrics_{stamp}.csv"
json_path = out_dir / f"test_metrics_summary_{stamp}.json"

with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

with open(json_path, "w") as f:
    json.dump({
        "macro_soft": macro_soft,
        "macro_hard@0.50": macro_hard,
        "micro_soft": micro_soft,
        "n_cases": len(rows),
        "csv": str(csv_path),
    }, f, indent=2)

print(f"\nSaved CSV  -> {csv_path}")
print(f"Saved JSON -> {json_path}")


[10/138] last case soft=0.8195 hard@0.50=0.8237
[20/138] last case soft=0.8492 hard@0.50=0.8516
[30/138] last case soft=0.7897 hard@0.50=0.7932
[40/138] last case soft=0.5236 hard@0.50=0.5261
[50/138] last case soft=0.6927 hard@0.50=0.6957
[60/138] last case soft=0.6711 hard@0.50=0.6754
[70/138] last case soft=0.7172 hard@0.50=0.7217
[80/138] last case soft=0.4940 hard@0.50=0.5029
[90/138] last case soft=0.6561 hard@0.50=0.6604
[100/138] last case soft=0.0086 hard@0.50=0.0089
[110/138] last case soft=0.1217 hard@0.50=0.1236
[120/138] last case soft=0.2790 hard@0.50=0.2840
[130/138] last case soft=0.1385 hard@0.50=0.1425
[138/138] last case soft=0.0001 hard@0.50=0.0000

=== HELD-OUT SUMMARY ===
Macro soft Dice        : 0.4316
Macro hard Dice @0.50  : 0.4362
Micro soft Dice        : 0.6286
N cases: 138

Bottom 10 (soft Dice):
  118 | 0.0000 | vox=   1646 | sub-r038s065_ses-1_T1w_MNI_norm.nii.gz
  138 | 0.0001 | vox=   1814 | sub-r048s043_ses-1_T1w_MNI_norm.nii.gz
   25 | 0.0001 | vox=   

In [2]:
# --- Cohort + size-bin summary without pandas ---
import csv, re, math, os

CSV = "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v2/runs/20251107_103356/test_eval/test_metrics_20251107_151557.csv"

def cohort_from_name(name: str) -> str:
    m = re.match(r"sub-([A-Za-z]+)", os.path.basename(name))
    return (m.group(1) if m else "unknown").lower()

def mean(xs): 
    xs = [x for x in xs if not math.isnan(x)]
    return sum(xs)/len(xs) if xs else float("nan")

rows = []
with open(CSV, newline="") as f:
    r = csv.DictReader(f)
    for row in r:
        rows.append({
            "image": row["image"],
            "cohort": cohort_from_name(row["image"]),
            "dice_soft": float(row["dice_soft"]),
            "dice_hard": float(row["dice_hard@0.50"]),
            "vox": int(row["lesion_voxels"]),
        })

# Cohort breakdown
by_cohort = {}
for x in rows:
    by_cohort.setdefault(x["cohort"], []).append(x)
print("=== Cohort summary ===")
for c, xs in sorted(by_cohort.items()):
    print(f"{c:>6s} | n={len(xs):3d} | mean dice_soft={mean([i['dice_soft'] for i in xs]):.4f} | "
          f"median vox={sorted(i['vox'] for i in xs)[len(xs)//2]}")

# Size bins
bins = [(0,2500),(2500,10000),(10000,30000),(30000,60000),(60000,10**9)]
print("\n=== Size-bin summary (voxels) ===")
for lo,hi in bins:
    xs = [x for x in rows if lo <= x["vox"] < hi]
    print(f"[{lo:6d},{hi:6d}) | n={len(xs):3d} | mean dice_soft={mean([i['dice_soft'] for i in xs]):.4f}")


=== Cohort summary ===
     m | n= 52 | mean dice_soft=0.5799 | median vox=46927
     r | n= 86 | mean dice_soft=0.3419 | median vox=9790

=== Size-bin summary (voxels) ===
[     0,  2500) | n= 28 | mean dice_soft=0.0914
[  2500, 10000) | n= 20 | mean dice_soft=0.2895
[ 10000, 30000) | n= 32 | mean dice_soft=0.4364
[ 30000, 60000) | n= 34 | mean dice_soft=0.5650
[ 60000,1000000000) | n= 24 | mean dice_soft=0.7512
